In [15]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape
from shapely import wkt
alt.data_transformers.disable_max_rows()
from shapely.ops import linemerge, substring, unary_union


In [16]:
nodes = pd.read_csv("Nodes/FGC.csv")
nodes['geometry']= nodes['geometry'].apply(wkt.loads)
nodes = gpd.GeoDataFrame(nodes, geometry='geometry', crs="EPSG:4326")

In [17]:
with open('data/FGC/fcg_trajectories.json') as f:
    data = json.load(f)
fgc_traj = []
for stop in data['features']:
    properties = stop.get('properties', {})
    route_id = properties.get('route_id')
    route_name = properties.get('route_long_name')

    
    fgc_traj.append({
        'route_id': route_id,
        'route_name': route_name,
        'geometry': shape(stop['geometry'])

    })  
geo_df_fgc_traj = gpd.GeoDataFrame(fgc_traj, crs="EPSG:4326")
valid = ['L6', 'L7', 'L8', 'L12','S1']
geo_df_fgc_traj = geo_df_fgc_traj[geo_df_fgc_traj['route_id'].isin(valid)]
geo_df_fgc_traj

,route_id,route_name,geometry
0,L6,Barcelona Pl. Catalunya - Sarrià,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."
9,L7,Barcelona Pl. Catalunya - Av Tibidabo,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."
12,S1,Barcelona Pl. Catalunya - Terrassa Nacions Unides,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."
17,L8,Barcelona Pl. Espanya - Molí Nou,"MULTILINESTRING ((2.14847 41.37452, 2.14466 41..."
18,L12,Sarrià - Reina Elisenda,"MULTILINESTRING ((2.12557 41.3985, 2.12478 41...."


In [18]:
def break_trajectory(trajectory_df, stops):

    line = trajectory_df.geometry.iloc[0]
    #line = trajectory_df[trajectory_df['route_id'] == 'L6'].geometry.iloc[0]

    flattened_line = unary_union(line) 
    merged_line = linemerge(flattened_line)

    if merged_line.geom_type == 'MultiLineString':
        line_s = max(merged_line.geoms, key=lambda x: x.length)
    else:
        line_s = merged_line

    stop_list = []
    for idx, row in stops.iterrows():
        dist = line_s.project(row.geometry)
        stop_list.append({'name': row['name'], 'id': row['id'], 'dist': dist})
    sorted_stops = sorted(stop_list, key=lambda x: x['dist'])

    # 4. Create segments
    segments_data = []
    for i in range(len(sorted_stops) - 1):
        origin = sorted_stops[i]
        destination = sorted_stops[i+1]
        
        # Check for zero length (stops at the same location)
        if abs(destination['dist'] - origin['dist']) < 1e-7:
            continue
            
        seg_geom = substring(line_s, origin['dist'], destination['dist'])
        
        segments_data.append({
            'origen': origin['id'],
            'dest': destination['id'],
            'tram': f"{origin['name']} - {destination['name']}",
            'linia': trajectory_df['route_id'].iloc[0],
            'type' : 'FGC',
            'geometry': seg_geom,
        })

    segments_gdf = gpd.GeoDataFrame(segments_data, crs="EPSG:4326")

    return segments_gdf



In [19]:
fgc_edges = pd.DataFrame()
for route in valid:
    segments_gdf = break_trajectory(geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == route], nodes[nodes['linia'] == route])
    fgc_edges = pd.concat([fgc_edges, segments_gdf], ignore_index=True)

In [20]:
fgc_edges.to_crs('EPSG:25831', inplace=True)
fgc_edges['length'] = fgc_edges['geometry'].length 
fgc_edges['speed'] = 25 /3.6 # 25 kmh to ms like in the paper
fgc_edges['time'] = pd.to_timedelta(
    fgc_edges['length'] / fgc_edges['speed'],
    unit='s'
)
fgc_edges['time'] = fgc_edges['time'].apply(
    lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
)
fgc_edges['directed'] = False
fgc_edges.to_crs('EPSG:4326', inplace=True)
fgc_edges = fgc_edges[['origen', 'dest', 'tram', 'linia', 'type', 'length', 'speed', 'time', 'directed', 'geometry']]
fgc_edges

,origen,dest,tram,linia,type,length,speed,time,directed,geometry
0,F-L6-PC,F-L6-PR,Catalunya - Diagonal,L6,FGC,1217.889670,6.944444,02:55,False,"LINESTRING (2.16872 41.38562, 2.16781 41.3857,..."
1,F-L6-PR,F-L6-GR,Diagonal - Gràcia,L6,FGC,840.644667,6.944444,02:01,False,"LINESTRING (2.15805 41.39284, 2.15615 41.39429..."
2,F-L6-GR,F-L6-SG,Gràcia - Sant Gervasi,L6,FGC,556.767947,6.944444,01:20,False,"LINESTRING (2.15266 41.3991, 2.15156 41.4002, ..."
3,F-L6-SG,F-L6-MN,Sant Gervasi - Muntaner,L6,FGC,496.060696,6.944444,01:11,False,"LINESTRING (2.14712 41.40113, 2.1467 41.40104,..."
4,F-L6-MN,F-L6-BN,Muntaner - La Bonanova,L6,FGC,507.685031,6.944444,01:13,False,"LINESTRING (2.14234 41.39854, 2.14211 41.39844..."
5,F-L6-BN,F-L6-TT,La Bonanova - Les Tres Torres,L6,FGC,469.992809,6.944444,01:07,False,"LINESTRING (2.13643 41.39786, 2.13618 41.39786..."
6,F-L6-TT,F-L6-SR,Les Tres Torres - Sarrià,L6,FGC,378.811513,6.944444,00:54,False,"LINESTRING (2.13081 41.39785, 2.13028 41.39785..."
7,F-L7-PC,F-L7-PR,Catalunya - Diagonal,L7,FGC,1217.995375,6.944444,02:55,False,"LINESTRING (2.16872 41.38562, 2.16781 41.3857,..."
8,F-L7-PR,F-L7-GR,Diagonal - Gràcia,L7,FGC,840.651150,6.944444,02:01,False,"LINESTRING (2.15805 41.39284, 2.15615 41.39429..."
9,F-L7-GR,F-L7-PM,Gràcia - Pl. Molina,L7,FGC,602.608604,6.944444,01:26,False,"LINESTRING (2.15266 41.3991, 2.15156 41.4002, ..."


In [21]:
fgc_edges.to_csv("Edges/FGC.csv", index=False)

In [14]:
lines = alt.Chart(fgc_edges[fgc_edges['linia'] == 'L8']).mark_geoshape(
    filled=False, 
    strokeWidth=4
).encode(
    color=alt.Color('tram:N', legend=alt.Legend(title="Metro Segments")),
    tooltip=['tram:N']
)

points = alt.Chart(nodes[nodes['linia'] == 'L8']).mark_geoshape(size=0, color='red').encode(tooltip=['name:N'])
plot = (lines + points).project('mercator').properties(width=800, height=600)
plot
    

alt.LayerChart(...)